# **Data cleaning**

We load the merged raw *Global Wellbeing, Sustainability and Resource Use Dataset*. 

We screen the data and look for missing values, specifically trying to identify countries with a substantial amount of missing values in the key variables:

| Key variables | Description | Source(s) |
|---|---|---|
| `happiness_index` | Measure of Subjective Wellbeing | World Happiness Report (https://www.kaggle.com/datasets/simonaasm/world-happiness-index-by-reports-2013-2023) |
| `gini_index` | Measure of income inequality | World Bank GINI Index (https://data.worldbank.org/indicator/SI.POV.GINI) |
| `consumption_co2_per_capita` | Consumption-based CO₂ emissions per capita | Our World in Data CO₂ (https://github.com/owid/co2-data) |
| `co2_per_capita` | Production-based CO₂ emissions per capita | Our World in Data CO₂ Data (https://github.com/owid/co2-data) |
| `renewables_consumption` | Share of primary energy consumption from renewable sources | Our World in Data Energy (https://github.com/owid/energy-data) |
| `energy_per_capita` | Primary energy consumption per capita | Our World in Data Energy (https://github.com/owid/energy-data) (See below) |
| `material_footprint_per_capita` | Per capita material consiumption indicator | UN Human Development Reports (https://www.kaggle.com/datasets/iamsouravbanerjee/material-footprint-per-capita-by-country) |

This will help us identify countries that may be more reasonable to drop than keep and handle missing values of. Relative to our variables of interest they contain more noise than actual information.

In [1]:
import pandas as pd

In [2]:
# Make imports from the scr/ directory work.
import sys
from pathlib import Path

# Add project root to path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

In [3]:
from scr.io import load_csv
from scr.config import RAW_PATH

df_raw = load_csv(RAW_PATH)
display(df_raw.head())

df_raw.info()

,country,iso_code,year,co2_per_capita,consumption_co2_per_capita,energy_per_capita_x,temperature_change_from_co2,share_global_co2,land_use_change_co2_per_capita,population,...,renewables_consumption,happiness_index,happiness_index_rank,continent,hemisphere,human_development_groups,hdi_rank_2021,undp_developing_regions,material_footprint_per_capita,gini_index
0,Aruba,ABW,2013,8.395,NaN,47742.637,0.0,0.002,NaN,102570.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Aruba,ABW,2014,8.435,NaN,47990.926,0.0,0.002,NaN,103381.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Aruba,ABW,2015,8.615,NaN,48905.531,0.0,0.003,NaN,104200.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Aruba,ABW,2016,8.411,NaN,47619.418,0.0,0.002,NaN,104989.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Aruba,ABW,2017,8.420,NaN,49061.195,0.0,0.002,NaN,105737.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 1962 entries, 0 to 1961
Data columns (total 22 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   country                         1962 non-null   str    
 1   iso_code                        1962 non-null   str    
 2   year                            1962 non-null   int64  
 3   co2_per_capita                  1917 non-null   float64
 4   consumption_co2_per_capita      1079 non-null   float64
 5   energy_per_capita_x             1836 non-null   float64
 6   temperature_change_from_co2     1935 non-null   float64
 7   share_global_co2                1935 non-null   float64
 8   land_use_change_co2_per_capita  1746 non-null   float64
 9   population                      1845 non-null   float64
 10  gdp                             1476 non-null   float64
 11  energy_per_capita_y             1836 non-null   float64
 12  renewables_consumption          711 non-null 

We notice we have two energy_per_capita variables, _x and _y. These come from two different datasets preceding the merge into a single dataset.
1. We consider both to establish whether they are different in any way.
2. If different, we must establish a criterion for keeping one over the other - first consider if any of the two has more missing values.

In [4]:
print(df_raw['energy_per_capita_x'].isna().sum())
print(df_raw['energy_per_capita_y'].isna().sum())

# We now identify which countries and years have missing data in each variable

print(df_raw[df_raw['energy_per_capita_x'].isna()]['country'].unique())
print(df_raw[df_raw['energy_per_capita_y'].isna()]['country'].unique())

# Final check that the non-missing values are the same in both variables, so we can safely drop one of them.
# We do this by computing their differences in every row, storing the results into a set, and then checking if the set contains only zero.

print(set(df_raw[df_raw['energy_per_capita_x'].notna()]['energy_per_capita_x'] - df_raw[df_raw['energy_per_capita_y'].notna()]['energy_per_capita_y']))

# There seem to be some differences. 
# We must find which values are different between the two variables 
# and print them as a list indexed by country and year 
# in order to understand where the differences are.

# Find rows where both values exist AND they're different
mask = (df_raw['energy_per_capita_x'].notna() & 
        df_raw['energy_per_capita_y'].notna() & 
        (df_raw['energy_per_capita_x'] != df_raw['energy_per_capita_y']))

print(df_raw[mask][['iso_code', 'year', 'energy_per_capita_x', 'energy_per_capita_y']])


126
126
<StringArray>
[                       'Anguilla',                         'Andorra',
                      'Antarctica', 'Bonaire Sint Eustatius and Saba',
                         'Curacao',                'Christmas Island',
                   'Liechtenstein',                          'Monaco',
                'Marshall Islands',                           'Palau',
                      'San Marino',       'Sint Maarten (Dutch part)',
                         'Vatican',               'Wallis and Futuna']
Length: 14, dtype: str
<StringArray>
[                       'Anguilla',                         'Andorra',
                      'Antarctica', 'Bonaire Sint Eustatius and Saba',
                         'Curacao',                'Christmas Island',
                   'Liechtenstein',                          'Monaco',
                'Marshall Islands',                           'Palau',
                      'San Marino',       'Sint Maarten (Dutch part)',
                  

So all differences come from a single country, Togo (iso_code: TGO), where `energy_per_capita_y > energy_per_capita_x`

Looking at the source of the original datasets, the energy dataset (y) has been updated more recently than the CO2 dataset (x) - 3 weeks vs 5 months. This aligns with the observation prior to merging, that for the same countries, the energy dataset had greater richness in the population and gdp variables - which where consequently kept to maximise the availability of real information.

Following along these lines, we make the choice to keep the y-variable, from energy data, rather than taking an average between the two.

In [5]:
# We must now rename the energy_per_capita_y variable to energy_per_capita, and drop the _x version
df_raw = df_raw.rename(columns={'energy_per_capita_y': 'energy_per_capita'})
df_raw = df_raw.drop(columns=['energy_per_capita_x'])

We select key variables and identify the number of missing values per iso_code/country (where there are any) aggregated over the years.

To aid with this process in a managable manner given the large number of countries, we build a function to:
1. Compute missing values by country-year.
2. Identify country-years with more than N missing variables.
3. Extract the affected countries.
4. Return the country-level missingness summary for only those countries.

In [11]:
from scr.utils import countries_with_missing_vars

key_vars = [
    "happiness_index", 
    "gini_index", 
    "material_footprint_per_capita", 
    "consumption_co2_per_capita", 
    "co2_per_capita", 
    "energy_per_capita", 
    "renewables_consumption"
    ]

subset_missing_6 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=6
)

display(subset_missing_6)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,
Antarctica,9,9,9,9,9,9,9
Christmas Island,9,9,9,9,9,9,9
Monaco,9,9,9,9,9,9,9
San Marino,9,9,9,9,9,9,9
Vatican,9,9,9,9,9,9,9
